In [2]:
pip install psycopg[binary]

   ---------------------------------------- 0.0/3.8 MB ? eta -:--:--
   ---------------------------------------- 3.8/3.8 MB 32.6 MB/s  0:00:00

   ---------------------------------------- 0/2 [psycopg-binary]
   -------------------- ------------------- 1/2 [psycopg]
   -------------------- ------------------- 1/2 [psycopg]
   -------------------- ------------------- 1/2 [psycopg]
   -------------------- ------------------- 1/2 [psycopg]
   ---------------------------------------- 2/2 [psycopg]

Note: you may need to restart the kernel to use updated packages.


In [3]:
import os
import pandas as pd
from sqlalchemy import create_engine

# Database connection parameters
DB_USER = os.getenv("POSTGRES_USER", "postgres")
DB_PASSWORD = os.getenv("POSTGRES_PASSWORD", "senhapost")
DB_HOST = os.getenv("POSTGRES_HOST", "localhost")
DB_PORT = os.getenv("POSTGRES_PORT", "5432")
DB_NAME = "loadsmart_dev"
SCHEMA_NAME = "analytics_analytics"

# Create SQLAlchemy Engine
connection_url = f"postgresql://{DB_USER}:{DB_PASSWORD}@{DB_HOST}:{DB_PORT}/{DB_NAME}"
engine = create_engine(connection_url)

print("Database engine created successfully.")

Database engine created successfully.


In [4]:
# SQL Query using window functions (DATE_TRUNC) to dynamically target the last available month
# for delivered loads (is_delivered = TRUE).

query = f"""
WITH delivered_loads AS (
    SELECT 
        f.loadsmart_id,
        s.shipper_name,
        f.delivery_date,
        l.pickup_city,
        l.pickup_state,
        l.delivery_city,
        l.delivery_state,
        f.book_price,
        c.carrier_name,
        DATE_TRUNC('month', f.delivery_date) AS delivery_month
    FROM {SCHEMA_NAME}.fact_load f
    INNER JOIN {SCHEMA_NAME}.dim_shipper s ON f.shipper_id = s.shipper_id
    INNER JOIN {SCHEMA_NAME}.dim_carrier c ON f.carrier_id = c.carrier_id
    INNER JOIN {SCHEMA_NAME}.dim_lane l ON f.lane_id = l.lane_id
    WHERE f.is_delivered = TRUE
      AND f.delivery_date IS NOT NULL
),
latest_month AS (
    SELECT MAX(delivery_month) AS max_month 
    FROM delivered_loads
)
SELECT 
    dl.loadsmart_id,
    dl.shipper_name,
    dl.delivery_date,
    dl.pickup_city,
    dl.pickup_state,
    dl.delivery_city,
    dl.delivery_state,
    dl.book_price,
    dl.carrier_name
FROM delivered_loads dl
CROSS JOIN latest_month lm
WHERE dl.delivery_month = lm.max_month
ORDER BY dl.delivery_date ASC, dl.loadsmart_id ASC;
"""

In [5]:
# Read data into DataFrame
df = pd.read_sql(query, engine)

# Preview results
print(f"Total records retrieved: {len(df)}")
df.head()

Total records retrieved: 1


,loadsmart_id,shipper_name,delivery_date,pickup_city,pickup_state,delivery_city,delivery_state,book_price,carrier_name
0,206665369,Shipper 758,2025-03-15 13:50:00,Lodi,CA,Pacific,WA,1955.56,Carrier 86454


In [6]:
# Output CSV file path
output_filename = "delivered_loads_last_available_month.csv"

# Export to CSV
df.to_csv(
    output_filename,
    index=False,
    columns=[
        "loadsmart_id",
        "shipper_name",
        "delivery_date",
        "pickup_city",
        "pickup_state",
        "delivery_city",
        "delivery_state",
        "book_price",
        "carrier_name",
    ],
)

print(f"Successfully exported {len(df)} rows to '{output_filename}'.")

Successfully exported 1 rows to 'delivered_loads_last_available_month.csv'.
